# Notebook 11: Training Phase 1 - Projecteur Alignment

## Objectif
Premier phase d'entraînement du modèle Qwen2.5-VL-7B pour CassavaVLM:
- **Phase 1**: Aligner le projecteur vision→language (1 epoch, ~3-4h)
- Geler vision encoder, Q-Former, LLM
- Entraîner uniquement le projecteur pour alignment rapide
- Utiliser weighted loss pour compenser l'imbalance 16.7:1
- Monitoring complet avec TensorBoard & WandB

## Configuration Hardware
- **GPU**: NVIDIA RTX 5090 32GB VRAM
- **Mixed Precision**: bfloat16 (bf16)
- **Gradient Checkpointing**: Activé pour économiser VRAM

## Inputs
- Modèle base: Qwen2.5-VL-7B-Instruct
- LoRA config: du Notebook 09
- Class weights: du Notebook 10 (16.7:1 compensated)
- Dataset: 30,491 images JSONL format

## Outputs
- Checkpoint Phase 1: `models/checkpoints/phase1_projector/`
- Logs TensorBoard: `outputs/logs/phase1/`
- Métriques WandB: Online dashboard

## Durée Estimée
**~3-4 heures** sur RTX 5090 (1 epoch, 24,392 images train)

## 1. Imports & Setup

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import json
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Transformers & PEFT
from transformers import AutoModelForVision2Seq, AutoProcessor
from peft import LoraConfig, get_peft_model, TaskType

# WandB (optionnel)
try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False
    print("⚠️ WandB non disponible, utilisation de TensorBoard uniquement")

# Configuration
PROJECT_ROOT = Path("/home/hounfodjidagba/learning/cassava")
VLM_DATASET_DIR = PROJECT_ROOT / "outputs" / "vlm_dataset"
WEIGHTS_DIR = PROJECT_ROOT / "outputs" / "augmentation_weights"
CHECKPOINT_DIR = PROJECT_ROOT / "models" / "checkpoints" / "phase1_projector"
LOGS_DIR = PROJECT_ROOT / "outputs" / "logs" / "phase1"

# Créer les répertoires
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("="*70)
print("CONFIGURATION TRAINING PHASE 1")
print("="*70)
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"\nDirectories:")
print(f"  Dataset: {VLM_DATASET_DIR}")
print(f"  Checkpoints: {CHECKPOINT_DIR}")
print(f"  Logs: {LOGS_DIR}")
print("="*70)

CONFIGURATION TRAINING PHASE 1
Device: cuda
GPU: NVIDIA GeForce RTX 5090
VRAM: 33.67 GB
PyTorch version: 2.9.1+cu128
CUDA version: 12.8

Directories:
  Dataset: /home/hounfodjidagba/learning/cassava/outputs/vlm_dataset
  Checkpoints: /home/hounfodjidagba/learning/cassava/models/checkpoints/phase1_projector
  Logs: /home/hounfodjidagba/learning/cassava/outputs/logs/phase1


In [2]:
# Charger la config LoRA du Notebook 09
lora_config_path = PROJECT_ROOT / "outputs" / "lora_dataloader" / "lora_dataloader_config.json"
with open(lora_config_path, 'r') as f:
    config_data = json.load(f)
    lora_config_dict = config_data['lora_config']  # Accéder à la sous-clé


print("="*70)
print("CONFIGURATION LORA (du Notebook 09)")
print("="*70)
print(f"Rank (r): {lora_config_dict['r']}")
print(f"Alpha: {lora_config_dict['lora_alpha']}")
print(f"Dropout: {lora_config_dict['lora_dropout']}")
print(f"Target modules: {lora_config_dict['target_modules']}")
print(f"Task type: {lora_config_dict['task_type']}")
print("="*70)

# Charger les class weights du Notebook 10
class_weights_path = WEIGHTS_DIR / "class_weights.pt"
class_weights = torch.load(class_weights_path)

print("\n" + "="*70)
print("CLASS WEIGHTS (du Notebook 10)")
print("="*70)
print(f"Class weights tensor: {class_weights}")
print(f"Shape: {class_weights.shape}")
print(f"Device: {class_weights.device}")
print(f"\nRatio max/min: {class_weights.max() / class_weights.min():.1f}:1")
print("  → Compense le déséquilibre 16.7:1 des classes")
print("="*70)

# Charger dataset info
dataset_info_path = VLM_DATASET_DIR / "dataset_info.json"
with open(dataset_info_path, 'r') as f:
    dataset_info = json.load(f)

print("\n" + "="*70)
print("DATASET INFO")
print("="*70)
print(f"Total images: {dataset_info['total_images']:,}")
print(f"Train split: {dataset_info['splits']['train']} images")
print(f"Val split: {dataset_info['splits']['val']} images")
print(f"Test split: {dataset_info['splits']['test']} images")
print("="*70)

CONFIGURATION LORA (du Notebook 09)
Rank (r): 16
Alpha: 32
Dropout: 0.05
Target modules: ['o_proj', 'k_proj', 'q_proj', 'v_proj']
Task type: TaskType.CAUSAL_LM

CLASS WEIGHTS (du Notebook 10)
Class weights tensor: tensor([5.6101, 1.1759, 2.5558, 0.3362, 1.6499])
Shape: torch.Size([5])
Device: cpu

Ratio max/min: 16.7:1
  → Compense le déséquilibre 16.7:1 des classes

DATASET INFO
Total images: 30,491
Train split: {'num_samples': 24392, 'percentage': 79.99737627496638} images
Val split: {'num_samples': 3049, 'percentage': 9.999672034370798} images
Test split: {'num_samples': 3050, 'percentage': 10.002951690662819} images


## 3. Chargement du Modèle & Application LoRA

Charger Qwen2.5-VL-7B-Instruct et appliquer LoRA.

In [3]:
print("Chargement du modèle Qwen2.5-VL-7B-Instruct...")

model_name = "Qwen/Qwen2.5-VL-7B-Instruct"

# Charger le modèle base
model = AutoModelForVision2Seq.from_pretrained(
    model_name,
    dtype="auto",  # Utilise bfloat16 automatiquement sur RTX 5090
    device_map="auto"
)

# Charger le processor
processor = AutoProcessor.from_pretrained(model_name)

print("\n✓ Modèle base chargé")

# Activer gradient checkpointing pour économiser VRAM
model.gradient_checkpointing_enable()
print("✓ Gradient checkpointing activé")

# Recréer la config LoRA
lora_config = LoraConfig(
    r=lora_config_dict['r'],
    lora_alpha=lora_config_dict['lora_alpha'],
    lora_dropout=lora_config_dict['lora_dropout'],
    bias=lora_config_dict['bias'],
    target_modules=lora_config_dict['target_modules'],
    task_type=TaskType.CAUSAL_LM
)

# Appliquer LoRA
model = get_peft_model(model, lora_config)

print("\n" + "="*70)
print("PARAMÈTRES ENTRAÎNABLES")
print("="*70)
model.print_trainable_parameters()
print("="*70)

Chargement du modèle Qwen2.5-VL-7B-Instruct...


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.



✓ Modèle base chargé
✓ Gradient checkpointing activé

PARAMÈTRES ENTRAÎNABLES
trainable params: 10,092,544 || all params: 8,302,259,200 || trainable%: 0.1216


## 4. Stratégie de Freezing Phase 1

**Phase 1**: Aligner uniquement le projecteur
- ✅ Geler: Vision encoder
- ✅ Geler: LLM (sauf LoRA adapters)
- ❌ Entraîner: Projecteur uniquement

In [4]:
print("="*70)
print("FREEZING STRATEGY - PHASE 1")
print("="*70)

# Pour Phase 1, on garde les adapters LoRA gelés aussi
# On entraîne SEULEMENT le projecteur
for name, param in model.named_parameters():
    # Geler tous les paramètres par défaut
    param.requires_grad = False
    
    # Dégeler uniquement le projecteur (merge_layer ou visual_projection)
    if 'visual' in name and ('proj' in name or 'merge' in name):
        param.requires_grad = True

# Compter les paramètres entraînables
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"\nParamètres entraînables: {trainable_params:,}")
print(f"Paramètres totaux: {total_params:,}")
print(f"Pourcentage entraînable: {100 * trainable_params / total_params:.4f}%")

# Afficher les modules entraînables
print("\nModules entraînables:")
trainable_modules = [name for name, param in model.named_parameters() if param.requires_grad]
for i, name in enumerate(trainable_modules[:10], 1):  # Afficher les 10 premiers
    print(f"  {i}. {name}")
if len(trainable_modules) > 10:
    print(f"  ... et {len(trainable_modules) - 10} autres")

print("\n⚠️ PHASE 1: Entraînement du projecteur uniquement")
print("   → Alignment rapide vision→language (1 epoch)")
print("="*70)

FREEZING STRATEGY - PHASE 1

Paramètres entraînables: 519,058,944
Paramètres totaux: 8,302,259,200
Pourcentage entraînable: 6.2520%

Modules entraînables:
  1. base_model.model.model.visual.patch_embed.proj.weight
  2. base_model.model.model.visual.blocks.0.attn.proj.weight
  3. base_model.model.model.visual.blocks.0.attn.proj.bias
  4. base_model.model.model.visual.blocks.0.mlp.gate_proj.weight
  5. base_model.model.model.visual.blocks.0.mlp.gate_proj.bias
  6. base_model.model.model.visual.blocks.0.mlp.up_proj.weight
  7. base_model.model.model.visual.blocks.0.mlp.up_proj.bias
  8. base_model.model.model.visual.blocks.0.mlp.down_proj.weight
  9. base_model.model.model.visual.blocks.0.mlp.down_proj.bias
  10. base_model.model.model.visual.blocks.1.attn.proj.weight
  ... et 252 autres

⚠️ PHASE 1: Entraînement du projecteur uniquement
   → Alignment rapide vision→language (1 epoch)


In [5]:
# Importer le dataset du notebook 09
import sys
from PIL import Image
from torch.utils.data import Dataset
from qwen_vl_utils import process_vision_info

class CassavaVLMDataset(Dataset):
    """Dataset pour Qwen2.5-VL avec format JSONL."""
    
    def __init__(self, jsonl_path, image_base_path, processor, max_length=512):
        self.samples = []
        self.image_base_path = Path(image_base_path)
        self.processor = processor
        self.max_length = max_length
        
        # Charger les samples du JSONL
        with open(jsonl_path, 'r') as f:
            for line in f:
                self.samples.append(json.loads(line))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # Charger l'image
        image_path = self.image_base_path / sample['image']
        image = Image.open(image_path).convert('RGB')
        
        # Construire les messages au format Qwen
        messages = []
        for conv in sample['conversations']:
            role = "user" if conv['from'] == "human" else "assistant"
            content_text = conv['value']
            
            if '<image>' in content_text:
                # Message avec image
                text = content_text.replace('<image>\n', '').replace('<image>', '')
                content = [
                    {"type": "image", "image": image},
                    {"type": "text", "text": text}
                ]
            else:
                # Message texte seul
                content = content_text
            
            messages.append({"role": role, "content": content})
        
        # Traiter avec le processor Qwen
        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        image_inputs, video_inputs = process_vision_info(messages)
        
        inputs = self.processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        # Extraire les tensors
        input_ids = inputs['input_ids'].squeeze(0)
        attention_mask = inputs['attention_mask'].squeeze(0)
        pixel_values = inputs.get('pixel_values', None)
        image_grid_thw = inputs.get('image_grid_thw', None)  # IMPORTANT: paramètre requis par Qwen2.5-VL
        
        if pixel_values is not None:
            pixel_values = pixel_values.squeeze(0)
        if image_grid_thw is not None:
            image_grid_thw = image_grid_thw.squeeze(0)
        
        # Labels = input_ids (autoregressive training)
        labels = input_ids.clone()
        
        # Extraire metadata
        label = sample['metadata']['label']
        class_short = sample['metadata']['class_short']
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'pixel_values': pixel_values,
            'image_grid_thw': image_grid_thw,
            'labels': labels,
            'label': label,
            'class_short': class_short,
            'image_id': sample['id']
        }

def collate_fn(batch):
    """Collate function pour batching."""
    # Trouver les longueurs max
    max_length = max(item['input_ids'].size(0) for item in batch)
    
    # Préparer les tensors
    input_ids = []
    attention_masks = []
    labels_list = []
    pixel_values_list = []
    image_grid_thw_list = []
    label_list = []
    class_short_list = []
    image_id_list = []
    
    for item in batch:
        # Pad input_ids et attention_mask
        seq_len = item['input_ids'].size(0)
        pad_len = max_length - seq_len
        
        input_ids.append(torch.cat([
            item['input_ids'],
            torch.zeros(pad_len, dtype=torch.long)
        ]))
        
        attention_masks.append(torch.cat([
            item['attention_mask'],
            torch.zeros(pad_len, dtype=torch.long)
        ]))
        
        # Pad labels avec -100 (ignore index)
        labels_list.append(torch.cat([
            item['labels'],
            torch.full((pad_len,), -100, dtype=torch.long)
        ]))
        
        if item['pixel_values'] is not None:
            pixel_values_list.append(item['pixel_values'])
        
        if item['image_grid_thw'] is not None:
            image_grid_thw_list.append(item['image_grid_thw'])
        
        label_list.append(item['label'])
        class_short_list.append(item['class_short'])
        image_id_list.append(item['image_id'])
    
    # Stack tensors
    batched = {
        'input_ids': torch.stack(input_ids),
        'attention_mask': torch.stack(attention_masks),
        'labels': torch.stack(labels_list),
        'label': torch.tensor(label_list, dtype=torch.long),
        'class_short': class_short_list,
        'image_id': image_id_list
    }
    
    if pixel_values_list:
        batched['pixel_values'] = torch.stack(pixel_values_list)
    
    if image_grid_thw_list:
        batched['image_grid_thw'] = torch.stack(image_grid_thw_list)
    
    return batched

print("✓ Classes Dataset et collate_fn définies")

✓ Classes Dataset et collate_fn définies


## 5. Configuration du Loss avec Class Weights

Utiliser weighted CrossEntropyLoss pour compenser l'imbalance.

In [6]:
# Déplacer class weights sur le device
class_weights = class_weights.to(device)

print("="*70)
print("LOSS CONFIGURATION")
print("="*70)
print(f"Loss function: Weighted CrossEntropyLoss")
print(f"Class weights: {class_weights}")
print(f"Device: {class_weights.device}")
print("\n⚠️ Note: Pour VLM, nous utilisons le loss natif du modèle")
print("   Les class weights seront appliqués via sample weighting")
print("="*70)

LOSS CONFIGURATION
Loss function: Weighted CrossEntropyLoss
Class weights: tensor([5.6101, 1.1759, 2.5558, 0.3362, 1.6499], device='cuda:0')
Device: cuda:0

⚠️ Note: Pour VLM, nous utilisons le loss natif du modèle
   Les class weights seront appliqués via sample weighting


## 6. Configuration de l'Entraînement

In [7]:
# Hyperparamètres Phase 1
training_config = {
    # Training
    'num_epochs': 1,
    'batch_size': 2,  # Petit batch pour VRAM
    'gradient_accumulation_steps': 8,  # Batch effectif = 16
    'learning_rate': 1e-3,  # LR élevé pour alignment rapide
    'weight_decay': 0.01,
    'warmup_steps': 100,
    'max_grad_norm': 1.0,  # Gradient clipping
    
    # Optimizer
    'optimizer': 'AdamW',
    'betas': (0.9, 0.999),
    'eps': 1e-8,
    
    # Scheduler
    'scheduler': 'cosine',
    
    # Mixed precision
    'mixed_precision': 'bf16',  # bfloat16 pour RTX 5090
    
    # Logging
    'log_every_n_steps': 10,
    'eval_every_n_steps': 500,
    'save_every_n_steps': 1000,
    
    # Other
    'seed': 42,
}

print("="*70)
print("HYPERPARAMÈTRES PHASE 1")
print("="*70)
for key, value in training_config.items():
    print(f"{key:30s}: {value}")
print("="*70)

# Set seed
torch.manual_seed(training_config['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(training_config['seed'])

HYPERPARAMÈTRES PHASE 1
num_epochs                    : 1
batch_size                    : 2
gradient_accumulation_steps   : 8
learning_rate                 : 0.001
weight_decay                  : 0.01
warmup_steps                  : 100
max_grad_norm                 : 1.0
optimizer                     : AdamW
betas                         : (0.9, 0.999)
eps                           : 1e-08
scheduler                     : cosine
mixed_precision               : bf16
log_every_n_steps             : 10
eval_every_n_steps            : 500
save_every_n_steps            : 1000
seed                          : 42


In [8]:
from torch.cuda.amp import autocast

def train_epoch(model, train_loader, optimizer, scheduler, epoch, config):
    """Entraîner une epoch."""
    model.train()
    
    total_loss = 0
    optimizer.zero_grad()
    
    progress_bar = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc=f"Epoch {epoch+1}/{config['num_epochs']}"
    )
    
    for step, batch in progress_bar:
        # Déplacer sur device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Gérer pixel_values et image_grid_thw (optionnels)
        pixel_values = batch.get('pixel_values')
        if pixel_values is not None:
            pixel_values = pixel_values.to(device)
        
        image_grid_thw = batch.get('image_grid_thw')
        if image_grid_thw is not None:
            image_grid_thw = image_grid_thw.to(device)
        
        # Forward pass avec mixed precision (bfloat16 - pas besoin de GradScaler)
        with autocast(dtype=torch.bfloat16):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                image_grid_thw=image_grid_thw,
                labels=labels
            )
            loss = outputs.loss
            
            # Normaliser par gradient accumulation
            loss = loss / config['gradient_accumulation_steps']
        
        # Backward (pas de scaling avec bfloat16)
        loss.backward()
        
        # Update weights chaque gradient_accumulation_steps
        if (step + 1) % config['gradient_accumulation_steps'] == 0:
            # Gradient clipping
            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                config['max_grad_norm']
            )
            
            # Optimizer step (pas de scaler avec bfloat16)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
            # Logging
            global_step = epoch * len(train_loader) + step
            current_lr = scheduler.get_last_lr()[0]
            
            if global_step % config['log_every_n_steps'] == 0:
                writer.add_scalar('train/loss', loss.item() * config['gradient_accumulation_steps'], global_step)
                writer.add_scalar('train/learning_rate', current_lr, global_step)
                writer.add_scalar('train/grad_norm', grad_norm.item(), global_step)
                
                if WANDB_AVAILABLE:
                    wandb.log({
                        'train/loss': loss.item() * config['gradient_accumulation_steps'],
                        'train/learning_rate': current_lr,
                        'train/grad_norm': grad_norm.item(),
                        'epoch': epoch
                    }, step=global_step)
        
        # Accumuler loss
        total_loss += loss.item()
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': f"{loss.item() * config['gradient_accumulation_steps']:.4f}",
            'lr': f"{scheduler.get_last_lr()[0]:.2e}"
        })
    
    avg_loss = total_loss / len(train_loader) * config['gradient_accumulation_steps']
    return avg_loss

print("✓ Fonction train_epoch définie (bfloat16 sans GradScaler)")

✓ Fonction train_epoch définie (bfloat16 sans GradScaler)


In [9]:
@torch.no_grad()
def validate(model, val_loader, epoch):
    """Valider le modèle."""
    model.eval()
    
    total_loss = 0
    
    progress_bar = tqdm(
        val_loader,
        desc=f"Validation Epoch {epoch+1}"
    )
    
    for batch in progress_bar:
        # Déplacer sur device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Gérer pixel_values et image_grid_thw (optionnels)
        pixel_values = batch.get('pixel_values')
        if pixel_values is not None:
            pixel_values = pixel_values.to(device)
        
        image_grid_thw = batch.get('image_grid_thw')
        if image_grid_thw is not None:
            image_grid_thw = image_grid_thw.to(device)
        
        # Forward pass
        with autocast(dtype=torch.bfloat16):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                image_grid_thw=image_grid_thw,
                labels=labels
            )
            loss = outputs.loss
        
        total_loss += loss.item()
        
        progress_bar.set_postfix({'val_loss': f"{loss.item():.4f}"})
    
    avg_loss = total_loss / len(val_loader)
    return avg_loss

print("✓ Fonction validate définie")

✓ Fonction validate définie


In [10]:
# Importer le dataset du notebook 09
import sys
from PIL import Image
from torch.utils.data import Dataset
from qwen_vl_utils import process_vision_info

class CassavaVLMDataset(Dataset):
    """Dataset pour Qwen2.5-VL avec format JSONL."""
    
    def __init__(self, jsonl_path, image_base_path, processor, max_length=512):
        self.samples = []
        self.image_base_path = Path(image_base_path)
        self.processor = processor
        self.max_length = max_length
        
        # Charger les samples du JSONL
        with open(jsonl_path, 'r') as f:
            for line in f:
                self.samples.append(json.loads(line))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # Charger l'image
        image_path = self.image_base_path / sample['image']
        image = Image.open(image_path).convert('RGB')
        
        # Construire les messages au format Qwen
        messages = []
        for conv in sample['conversations']:
            role = "user" if conv['from'] == "human" else "assistant"
            content_text = conv['value']
            
            if '<image>' in content_text:
                # Message avec image
                text = content_text.replace('<image>\n', '').replace('<image>', '')
                content = [
                    {"type": "image", "image": image},
                    {"type": "text", "text": text}
                ]
            else:
                # Message texte seul
                content = content_text
            
            messages.append({"role": role, "content": content})
        
        # Traiter avec le processor Qwen
        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        image_inputs, video_inputs = process_vision_info(messages)
        
        inputs = self.processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        # Extraire les tensors
        input_ids = inputs['input_ids'].squeeze(0)
        attention_mask = inputs['attention_mask'].squeeze(0)
        pixel_values = inputs.get('pixel_values', None)
        image_grid_thw = inputs.get('image_grid_thw', None)  # IMPORTANT: paramètre requis par Qwen2.5-VL
        
        if pixel_values is not None:
            pixel_values = pixel_values.squeeze(0)
        if image_grid_thw is not None:
            image_grid_thw = image_grid_thw.squeeze(0)
        
        # Labels = input_ids (autoregressive training)
        labels = input_ids.clone()
        
        # Extraire metadata
        label = sample['metadata']['label']
        class_short = sample['metadata']['class_short']
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'pixel_values': pixel_values,
            'image_grid_thw': image_grid_thw,
            'labels': labels,
            'label': label,
            'class_short': class_short,
            'image_id': sample['id']
        }

def collate_fn(batch):
    """Collate function pour batching."""
    # Trouver les longueurs max
    max_length = max(item['input_ids'].size(0) for item in batch)
    
    # Préparer les tensors
    input_ids = []
    attention_masks = []
    labels_list = []
    pixel_values_list = []
    image_grid_thw_list = []
    label_list = []
    class_short_list = []
    image_id_list = []
    
    for item in batch:
        # Pad input_ids et attention_mask
        seq_len = item['input_ids'].size(0)
        pad_len = max_length - seq_len
        
        input_ids.append(torch.cat([
            item['input_ids'],
            torch.zeros(pad_len, dtype=torch.long)
        ]))
        
        attention_masks.append(torch.cat([
            item['attention_mask'],
            torch.zeros(pad_len, dtype=torch.long)
        ]))
        
        # Pad labels avec -100 (ignore index)
        labels_list.append(torch.cat([
            item['labels'],
            torch.full((pad_len,), -100, dtype=torch.long)
        ]))
        
        if item['pixel_values'] is not None:
            pixel_values_list.append(item['pixel_values'])
        
        if item['image_grid_thw'] is not None:
            image_grid_thw_list.append(item['image_grid_thw'])
        
        label_list.append(item['label'])
        class_short_list.append(item['class_short'])
        image_id_list.append(item['image_id'])
    
    # Stack tensors
    batched = {
        'input_ids': torch.stack(input_ids),
        'attention_mask': torch.stack(attention_masks),
        'labels': torch.stack(labels_list),
        'label': torch.tensor(label_list, dtype=torch.long),
        'class_short': class_short_list,
        'image_id': image_id_list
    }
    
    if pixel_values_list:
        batched['pixel_values'] = torch.stack(pixel_values_list)
    
    if image_grid_thw_list:
        batched['image_grid_thw'] = torch.stack(image_grid_thw_list)
    
    return batched

print("✓ Classes Dataset et collate_fn définies")

✓ Classes Dataset et collate_fn définies


## 8. Création des DataLoaders

In [11]:
# Paths des JSONL
train_jsonl = VLM_DATASET_DIR / "train.jsonl"
val_jsonl = VLM_DATASET_DIR / "val.jsonl"

# Créer les datasets
print("Création des datasets...")
train_dataset = CassavaVLMDataset(
    jsonl_path=train_jsonl,
    image_base_path=VLM_DATASET_DIR,
    processor=processor,
    max_length=512
)

val_dataset = CassavaVLMDataset(
    jsonl_path=val_jsonl,
    image_base_path=VLM_DATASET_DIR,
    processor=processor,
    max_length=512
)

# Créer les dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=training_config['batch_size'],
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=training_config['batch_size'],
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True
)

print("="*70)
print("DATALOADERS CRÉÉS")
print("="*70)
print(f"Train dataset: {len(train_dataset):,} samples")
print(f"Val dataset: {len(val_dataset):,} samples")
print(f"Train batches: {len(train_loader):,}")
print(f"Val batches: {len(val_loader):,}")
print(f"Batch size: {training_config['batch_size']}")
print(f"Effective batch size: {training_config['batch_size'] * training_config['gradient_accumulation_steps']}")
print("="*70)

Création des datasets...
DATALOADERS CRÉÉS
Train dataset: 24,392 samples
Val dataset: 3,049 samples
Train batches: 12,196
Val batches: 1,525
Batch size: 2
Effective batch size: 16


## 9. Setup Optimizer & Scheduler

In [12]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

# Optimizer
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=training_config['learning_rate'],
    betas=training_config['betas'],
    eps=training_config['eps'],
    weight_decay=training_config['weight_decay']
)

# Calculer le nombre total de steps
num_training_steps = len(train_loader) * training_config['num_epochs'] // training_config['gradient_accumulation_steps']
warmup_steps = training_config['warmup_steps']

# Scheduler: Warmup + Cosine
warmup_scheduler = LinearLR(
    optimizer,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=warmup_steps
)

cosine_scheduler = CosineAnnealingLR(
    optimizer,
    T_max=num_training_steps - warmup_steps,
    eta_min=training_config['learning_rate'] * 0.1
)

scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_steps]
)

print("="*70)
print("OPTIMIZER & SCHEDULER")
print("="*70)
print(f"Optimizer: AdamW")
print(f"Learning rate: {training_config['learning_rate']}")
print(f"Weight decay: {training_config['weight_decay']}")
print(f"\nScheduler: Warmup ({warmup_steps} steps) + Cosine Annealing")
print(f"Total training steps: {num_training_steps:,}")
print(f"Warmup steps: {warmup_steps}")
print(f"Cosine steps: {num_training_steps - warmup_steps:,}")
print("="*70)

OPTIMIZER & SCHEDULER
Optimizer: AdamW
Learning rate: 0.001
Weight decay: 0.01

Scheduler: Warmup (100 steps) + Cosine Annealing
Total training steps: 1,524
Warmup steps: 100
Cosine steps: 1,424


## 10. Setup TensorBoard & WandB

In [13]:
# TensorBoard
writer = SummaryWriter(log_dir=LOGS_DIR)
print(f"✓ TensorBoard writer créé: {LOGS_DIR}")
print(f"  → Lancer avec: tensorboard --logdir {LOGS_DIR}")

# WandB (optionnel)
if WANDB_AVAILABLE:
    run_name = f"cassava_vlm_phase1_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    wandb.init(
        project="cassava-vlm",
        name=run_name,
        config={
            **training_config,
            'model': model_name,
            'phase': 'phase1_projector',
            'lora_r': lora_config_dict['r'],
            'lora_alpha': lora_config_dict['lora_alpha'],
            'trainable_params': trainable_params,
            'total_params': total_params
        }
    )
    print(f"✓ WandB run initialisé: {run_name}")
else:
    print("⚠️ WandB non disponible, logs TensorBoard uniquement")

✓ TensorBoard writer créé: /home/hounfodjidagba/learning/cassava/outputs/logs/phase1
  → Lancer avec: tensorboard --logdir /home/hounfodjidagba/learning/cassava/outputs/logs/phase1


wandb: Currently logged in as: hospicehounfodji (hospicehounfodji-epitech) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✓ WandB run initialisé: cassava_vlm_phase1_20260101_064447


## 11. Training Loop Phase 1

Entraînement du projecteur (1 epoch, ~3-4h).

In [14]:
from torch.cuda.amp import autocast

def train_epoch(model, train_loader, optimizer, scheduler, epoch, config):
    """Entraîner une epoch."""
    model.train()
    
    total_loss = 0
    optimizer.zero_grad()
    
    progress_bar = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc=f"Epoch {epoch+1}/{config['num_epochs']}"
    )
    
    for step, batch in progress_bar:
        # Déplacer sur device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Gérer pixel_values et image_grid_thw (optionnels)
        pixel_values = batch.get('pixel_values')
        if pixel_values is not None:
            pixel_values = pixel_values.to(device)
        
        image_grid_thw = batch.get('image_grid_thw')
        if image_grid_thw is not None:
            image_grid_thw = image_grid_thw.to(device)
        
        # Forward pass avec mixed precision (bfloat16 - pas besoin de GradScaler)
        with autocast(dtype=torch.bfloat16):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                image_grid_thw=image_grid_thw,
                labels=labels
            )
            loss = outputs.loss
            
            # Normaliser par gradient accumulation
            loss = loss / config['gradient_accumulation_steps']
        
        # Backward (pas de scaling avec bfloat16)
        loss.backward()
        
        # Update weights chaque gradient_accumulation_steps
        if (step + 1) % config['gradient_accumulation_steps'] == 0:
            # Gradient clipping
            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                config['max_grad_norm']
            )
            
            # Optimizer step (pas de scaler avec bfloat16)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
            # Logging
            global_step = epoch * len(train_loader) + step
            current_lr = scheduler.get_last_lr()[0]
            
            if global_step % config['log_every_n_steps'] == 0:
                writer.add_scalar('train/loss', loss.item() * config['gradient_accumulation_steps'], global_step)
                writer.add_scalar('train/learning_rate', current_lr, global_step)
                writer.add_scalar('train/grad_norm', grad_norm.item(), global_step)
                
                if WANDB_AVAILABLE:
                    wandb.log({
                        'train/loss': loss.item() * config['gradient_accumulation_steps'],
                        'train/learning_rate': current_lr,
                        'train/grad_norm': grad_norm.item(),
                        'epoch': epoch
                    }, step=global_step)
        
        # Accumuler loss
        total_loss += loss.item()
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': f"{loss.item() * config['gradient_accumulation_steps']:.4f}",
            'lr': f"{scheduler.get_last_lr()[0]:.2e}"
        })
    
    avg_loss = total_loss / len(train_loader) * config['gradient_accumulation_steps']
    return avg_loss

print("✓ Fonction train_epoch définie (bfloat16 sans GradScaler)")

✓ Fonction train_epoch définie (bfloat16 sans GradScaler)


## 12. Validation Function

In [15]:
@torch.no_grad()
def validate(model, val_loader, epoch):
    """Valider le modèle."""
    model.eval()
    
    total_loss = 0
    
    progress_bar = tqdm(
        val_loader,
        desc=f"Validation Epoch {epoch+1}"
    )
    
    for batch in progress_bar:
        # Déplacer sur device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Gérer pixel_values et image_grid_thw (optionnels)
        pixel_values = batch.get('pixel_values')
        if pixel_values is not None:
            pixel_values = pixel_values.to(device)
        
        image_grid_thw = batch.get('image_grid_thw')
        if image_grid_thw is not None:
            image_grid_thw = image_grid_thw.to(device)
        
        # Forward pass
        with autocast(dtype=torch.bfloat16):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                image_grid_thw=image_grid_thw,
                labels=labels
            )
            loss = outputs.loss
        
        total_loss += loss.item()
        
        progress_bar.set_postfix({'val_loss': f"{loss.item():.4f}"})
    
    avg_loss = total_loss / len(val_loader)
    return avg_loss

print("✓ Fonction validate définie")

✓ Fonction validate définie


## 13. Main Training Loop

⚠️ **ATTENTION**: Cette cellule va lancer l'entraînement qui durera **~3-4 heures**!

Assurez-vous:
- D'avoir suffisamment de temps
- Que le GPU est disponible
- Que le système ne va pas s'éteindre

In [16]:
print("="*70)
print("DÉBUT DE L'ENTRAÎNEMENT PHASE 1")
print("="*70)
print(f"Durée estimée: ~3-4 heures")
print(f"Epochs: {training_config['num_epochs']}")
print(f"Train batches: {len(train_loader):,}")
print(f"Effective batch size: {training_config['batch_size'] * training_config['gradient_accumulation_steps']}")
print("="*70)

start_time = time.time()
best_val_loss = float('inf')

for epoch in range(training_config['num_epochs']):
    print(f"\n{'='*70}")
    print(f"EPOCH {epoch + 1}/{training_config['num_epochs']}")
    print(f"{'='*70}")
    
    # Training (pas de scaler avec bfloat16)
    train_loss = train_epoch(
        model, train_loader, optimizer, scheduler,
        epoch, training_config
    )
    
    print(f"\nTrain Loss: {train_loss:.4f}")
    
    # Validation
    val_loss = validate(model, val_loader, epoch)
    print(f"Val Loss: {val_loss:.4f}")
    
    # Log epoch metrics
    writer.add_scalar('epoch/train_loss', train_loss, epoch)
    writer.add_scalar('epoch/val_loss', val_loss, epoch)
    
    if WANDB_AVAILABLE:
        wandb.log({
            'epoch/train_loss': train_loss,
            'epoch/val_loss': val_loss,
            'epoch': epoch
        })
    
    # Save checkpoint
    checkpoint_path = CHECKPOINT_DIR / f"epoch_{epoch+1}"
    checkpoint_path.mkdir(exist_ok=True)
    
    model.save_pretrained(checkpoint_path)
    processor.save_pretrained(checkpoint_path)
    
    # Save training state
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        'config': training_config
    }, checkpoint_path / 'training_state.pt')
    
    print(f"\n✓ Checkpoint sauvegardé: {checkpoint_path}")
    
    # Best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_checkpoint_path = CHECKPOINT_DIR / "best"
        best_checkpoint_path.mkdir(exist_ok=True)
        
        model.save_pretrained(best_checkpoint_path)
        processor.save_pretrained(best_checkpoint_path)
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'config': training_config
        }, best_checkpoint_path / 'training_state.pt')
        
        print(f"✓ Best checkpoint sauvegardé: {best_checkpoint_path}")

end_time = time.time()
training_duration = end_time - start_time

print("\n" + "="*70)
print("ENTRAÎNEMENT TERMINÉ")
print("="*70)
print(f"Durée totale: {training_duration / 3600:.2f}h")
print(f"Best val loss: {best_val_loss:.4f}")
print(f"Checkpoint final: {checkpoint_path}")
print(f"Best checkpoint: {best_checkpoint_path}")
print("="*70)

# Fermer les loggers
writer.close()
if WANDB_AVAILABLE:
    wandb.finish()

print("\n✓ Logs fermés")

DÉBUT DE L'ENTRAÎNEMENT PHASE 1
Durée estimée: ~3-4 heures
Epochs: 1
Train batches: 12,196
Effective batch size: 16

EPOCH 1/1


Epoch 1/1:   0%|          | 0/12196 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...



Train Loss: 6.5779


Validation Epoch 1:   0%|          | 0/1525 [00:00<?, ?it/s]

Val Loss: 6.5445

✓ Checkpoint sauvegardé: /home/hounfodjidagba/learning/cassava/models/checkpoints/phase1_projector/epoch_1
✓ Best checkpoint sauvegardé: /home/hounfodjidagba/learning/cassava/models/checkpoints/phase1_projector/best

ENTRAÎNEMENT TERMINÉ
Durée totale: 1.18h
Best val loss: 6.5445
Checkpoint final: /home/hounfodjidagba/learning/cassava/models/checkpoints/phase1_projector/epoch_1
Best checkpoint: /home/hounfodjidagba/learning/cassava/models/checkpoints/phase1_projector/best


epoch,▁
epoch/train_loss,▁
epoch/val_loss,▁
epoch,0
epoch/train_loss,6.57786
epoch/val_loss,6.54446



✓ Logs fermés


## 14. Résumé et Prochaines Étapes

### ✓ Accompli dans ce notebook:
1. **Modèle chargé** - Qwen2.5-VL-7B avec LoRA appliqué
2. **Freezing Phase 1** - Projecteur uniquement entraîné
3. **Weighted Loss** - Class weights appliqués (16.7:1 compensé)
4. **Training 1 epoch** - Alignment projecteur (~3-4h)
5. **Monitoring** - TensorBoard & WandB logs complets
6. **Checkpoints** - Sauvegardés (epoch_1 + best)

### 🎯 Résultats attendus:
- **Val loss**: Diminution significative vs baseline
- **Projecteur aligné**: Vision features → LLM space
- **Prêt pour Phase 2**: LoRA fine-tuning

### 📋 Prochaines étapes (Notebook 12 - Training Phase 2):
1. Charger checkpoint Phase 1
2. Dégeler les adapters LoRA
3. Fine-tuning complet (2-3 epochs, ~12-18h)
4. Early stopping si besoin
5. Target: 95-97% accuracy

### 📊 Fichiers générés:
- `models/checkpoints/phase1_projector/epoch_1/` - Checkpoint final
- `models/checkpoints/phase1_projector/best/` - Best checkpoint
- `outputs/logs/phase1/` - TensorBoard logs
- WandB dashboard online (si disponible)